# Phase 3: Training R(2+1)D-18

**Kaggle settings required:**
- **GPU T4 x2** (Settings → Accelerator → GPU T4 x2)
- Internet ON (to download Kinetics-400 pretrained weights)
- Add the output of Notebook 02 as an **input dataset**

**Architecture:** `r2plus1d_18` pretrained on Kinetics-400, fine-tuned on QEVD-92

**Progressive unfreezing schedule:**
- Epochs 1-5: only `fc` + `layer4`
- Epochs 6-10: also `layer3` (LR ×0.3)
- Epochs 11-15: also `layer2` (LR ×0.1)

**After running:** Download `best_r2plus1d_qevd.pth` for the Export notebook.

In [2]:
import os, json, time, warnings
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision.models.video import r2plus1d_18, R2Plus1D_18_Weights
from sklearn.model_selection import train_test_split
from collections import Counter
from tqdm import tqdm
import matplotlib
matplotlib.use('Agg')  # headless-safe
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

# =========================================================
# CONFIGURATION — update YOUR-PREPROCESSED-DATASET
# =========================================================
TENSORS_ROOT      = '/kaggle/input/notebooks/dadhichisarkershayon/preprocessing-2/preprocessed_tensors'
MANIFEST_PATH     = os.path.join(TENSORS_ROOT, 'manifest.jsonl')
CLASS_LABELS_PATH = os.path.join(TENSORS_ROOT, 'class_labels.json')
CLASS_MAP_PATH    = os.path.join(TENSORS_ROOT, 'class_map.json')

SAVE_DIR          = '/kaggle/working'
BEST_PATH         = os.path.join(SAVE_DIR, 'best_r2plus1d_qevd.pth')
LATEST_PATH       = os.path.join(SAVE_DIR, 'latest_checkpoint.pth')

EPOCHS            = 15
BATCH_SIZE        = 8     # safe for T4 16 GB; use 6 if you get OOM
INITIAL_LR        = 3e-4
WEIGHT_DECAY      = 1e-2
LABEL_SMOOTHING   = 0.1
GRAD_CLIP         = 1.0
NUM_WORKERS       = 4     # T4 kernel has 4 CPU cores

device   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
use_amp  = device.type == 'cuda'
print(f'🔥 Device: {device}')
if torch.cuda.is_available():
    print(f'   GPU: {torch.cuda.get_device_name(0)}')
    print(f'   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'   AMP: {use_amp}')

# Fail fast if the input dataset path is wrong
assert os.path.isdir(TENSORS_ROOT), (
    f'TENSORS_ROOT not found: {TENSORS_ROOT}\n'
    f'Update TENSORS_ROOT to match your Kaggle input dataset slug.'
)

🔥 Device: cuda
   GPU: Tesla T4
   VRAM: 15.6 GB
   AMP: True


In [3]:
# =========================================================
# LOAD CLASS MAPPING + MANIFEST
# =========================================================
with open(CLASS_LABELS_PATH) as f:
    class_labels = json.load(f)
with open(CLASS_MAP_PATH) as f:
    class_map = json.load(f)

num_classes = len(class_labels)
print(f'✅ {num_classes} classes discovered from metadata')

# =========================================================
# CRITICAL FIX: remap tensor paths
# =========================================================
MARKER = 'preprocessed_tensors'

def remap(old_path: str) -> str:
    # Normalize separators for robustness
    old_path = old_path.replace('\\', '/')
    idx = old_path.find(MARKER)
    if idx == -1:
        return old_path
    suffix = old_path[idx + len(MARKER):]   # e.g. /train/class/file.npy
    return TENSORS_ROOT.rstrip('/') + suffix

all_entries: list[dict] = []
with open(MANIFEST_PATH) as f:
    for line in f:
        line = line.strip()
        if not line: continue
        e = json.loads(line)
        if e['label'] not in class_map: continue
        e['tensor_path'] = remap(e['tensor_path'])
        all_entries.append(e)

assert all_entries, 'Manifest is empty!'
first = all_entries[0]['tensor_path']
print(f'✅ First entry remapped: {first}')

train_entries = [e for e in all_entries if e.get('split') == 'train']
val_entries   = [e for e in all_entries if e.get('split') == 'val']

if not val_entries:
    print('⚠️ No split found in manifest - falling back to 85/15 dynamic split')
    
    # 🔥 Robust dynamic split: duplicate 1-sample classes if they exist
    entries_to_split = []
    labels_for_split = []
    counts = Counter([e['label'] for e in all_entries])
    
    for e in all_entries:
        label = e['label']
        if counts[label] == 1:
            entries_to_split.extend([e, e])
            labels_for_split.extend([label, label])
            print(f'🔔 Duplicating rare class during fallback split: {label}')
        else:
            entries_to_split.append(e)
            labels_for_split.append(label)

    from sklearn.model_selection import train_test_split
    train_entries, val_entries = train_test_split(
        entries_to_split, test_size=0.15, random_state=42, stratify=labels_for_split
    )

print(f'📊 Train: {len(train_entries)} | Val: {len(val_entries)}')

✅ 91 classes discovered from metadata
✅ First entry remapped: /kaggle/input/notebooks/dadhichisarkershayon/preprocessing-2/preprocessed_tensors/train/air_jump_rope/00054022.npy
📊 Train: 5333 | Val: 1031


In [4]:
# =========================================================
# DATASET
# =========================================================
class QEVDDataset(Dataset):
    """
    Load pre-processed .npy tensors.
    Tensors are UN-normalized (float32 in [0,1]).
    We apply the competition mean/std here for training.
    """
    # Official competition normalization
    MEAN = torch.tensor([0.43216, 0.394666, 0.37645]).view(3, 1, 1, 1)   # (C,1,1,1)
    STD  = torch.tensor([0.22803, 0.22145,  0.216989]).view(3, 1, 1, 1)

    def __init__(self, entries, class_map, augment=False):
        self.entries   = entries
        self.class_map = class_map
        self.augment   = augment

    def __len__(self):
        return len(self.entries)

    def __getitem__(self, idx):
        e    = self.entries[idx]
        arr  = np.load(e['tensor_path'])                # (1, 3, 16, 112, 112)
        clip = torch.from_numpy(arr).squeeze(0)         # (3, 16, 112, 112)

        # Normalize
        clip = (clip - self.MEAN) / self.STD

        if self.augment:
            # Random horizontal flip
            if torch.rand(1).item() < 0.5:
                clip = torch.flip(clip, [-1])
            # Random temporal shift (±2 frames)
            if torch.rand(1).item() < 0.3:
                s = torch.randint(-2, 3, (1,)).item()
                if s:
                    clip = torch.roll(clip, shifts=s, dims=1)
            # Brightness jitter
            if torch.rand(1).item() < 0.3:
                clip = clip * (1.0 + (torch.rand(1).item() - 0.5) * 0.3)

        label = self.class_map[e['label']]
        return clip, label


print('✅ Dataset class defined')

# Quick smoke-test
ds_test = QEVDDataset([train_entries[0]], class_map)
clip_test, lbl_test = ds_test[0]
print(f'   clip shape : {clip_test.shape}')   # expect (3, 16, 112, 112)
print(f'   label      : {lbl_test}')

✅ Dataset class defined
   clip shape : torch.Size([3, 16, 112, 112])
   label      : 0


In [5]:
# =========================================================
# DATALOADERS — weighted sampler to handle class imbalance
# =========================================================
train_ds = QEVDDataset(train_entries, class_map, augment=True)
val_ds   = QEVDDataset(val_entries,   class_map, augment=False)

train_labels      = [class_map[e['label']] for e in train_entries]
class_counts      = np.bincount(train_labels, minlength=num_classes)
class_weights     = 1.0 / np.clip(class_counts, 1, None)
sample_weights    = [class_weights[l] for l in train_labels]

sampler = WeightedRandomSampler(
    torch.DoubleTensor(sample_weights), len(sample_weights), replacement=True
)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, sampler=sampler,
    num_workers=NUM_WORKERS, pin_memory=True, drop_last=True,
    persistent_workers=(NUM_WORKERS > 0),
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
    persistent_workers=(NUM_WORKERS > 0),
)

print(f'✅ Train loader: {len(train_loader)} batches')
print(f'✅ Val   loader: {len(val_loader)} batches')

✅ Train loader: 666 batches
✅ Val   loader: 129 batches


In [6]:
# =========================================================
# MODEL
# =========================================================
print('🚀 Building R(2+1)D-18 with Kinetics-400 weights...')
model = r2plus1d_18(weights=R2Plus1D_18_Weights.KINETICS400_V1)
model.fc = nn.Linear(model.fc.in_features, num_classes)
print(f'✅ FC head replaced: 512 → {num_classes}')

# ----- Progressive freeze helper -----
def set_phase(model, phase: int):
    """Freeze everything, then unfreeze progressively."""
    for p in model.parameters():
        p.requires_grad = False
    for p in model.fc.parameters():
        p.requires_grad = True
    if phase >= 1:
        for p in model.layer4.parameters(): p.requires_grad = True
    if phase >= 2:
        for p in model.layer3.parameters(): p.requires_grad = True
    if phase >= 3:
        for p in model.layer2.parameters(): p.requires_grad = True
    n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    n_total = sum(p.numel() for p in model.parameters())
    print(f'  Phase {phase}: trainable {n_train:,} / {n_total:,} ({100*n_train/n_total:.1f}%)')

set_phase(model, 1)
model = model.to(device)

🚀 Building R(2+1)D-18 with Kinetics-400 weights...
Downloading: "https://download.pytorch.org/models/r2plus1d_18-91a641e6.pth" to /root/.cache/torch/hub/checkpoints/r2plus1d_18-91a641e6.pth


100%|██████████| 120M/120M [00:00<00:00, 195MB/s]  


✅ FC head replaced: 512 → 91
  Phase 1: trainable 23,542,207 / 31,346,808 (75.1%)


In [7]:
# =========================================================
# OPTIMISER + SCHEDULER + AMP
# =========================================================
criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)

def make_opt(model, lr):
    return optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr, weight_decay=WEIGHT_DECAY
    )

optimizer  = make_opt(model, INITIAL_LR)
scheduler  = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
# GradScaler only when GPU is available — initialising it on CPU raises an error
scaler     = torch.amp.GradScaler('cuda') if use_amp else None

print('✅ Optimiser ready')
print(f'   AdamW  lr={INITIAL_LR}  wd={WEIGHT_DECAY}')
print(f'   CosineAnnealingLR  T_max={EPOCHS}')
print(f'   GradScaler: {scaler is not None}')

✅ Optimiser ready
   AdamW  lr=0.0003  wd=0.01
   CosineAnnealingLR  T_max=15
   GradScaler: True


In [8]:
# =========================================================
# TRAINING LOOP
# =========================================================
best_val_acc = 0.0
history = {'train_acc': [], 'val_acc': [], 'train_loss': [], 'val_loss': []}

print(f'\n🔥 Training {EPOCHS} epochs  |  batch={BATCH_SIZE}  |  AMP={use_amp}')
print('=' * 70)

for epoch in range(EPOCHS):
    t0 = time.time()

    # ---- Phase transitions ----
    if epoch == 5:
        print('\n🔓 Phase 2: unfreezing layer3')
        set_phase(model, 2)
        optimizer = make_opt(model, INITIAL_LR * 0.3)
        scheduler = optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=EPOCHS - epoch, eta_min=1e-6)
        scaler = torch.amp.GradScaler('cuda') if use_amp else None

    if epoch == 10:
        print('\n🔓 Phase 3: unfreezing layer2')
        set_phase(model, 3)
        optimizer = make_opt(model, INITIAL_LR * 0.1)
        scheduler = optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=EPOCHS - epoch, eta_min=1e-7)
        scaler = torch.amp.GradScaler('cuda') if use_amp else None

    # ---- TRAIN ----
    model.train()
    run_loss = run_correct = run_total = 0

    loop = tqdm(train_loader, desc=f'E{epoch+1:02d} train', leave=False)
    for x, y in loop:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)

        if use_amp:
            with torch.amp.autocast('cuda'):
                out  = model(x)
                loss = criterion(out, y)
            if torch.isnan(loss): continue
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()
        else:
            out  = model(x)
            loss = criterion(out, y)
            if torch.isnan(loss): continue
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()

        run_loss    += loss.item() * x.size(0)
        run_correct += out.argmax(1).eq(y).sum().item()
        run_total   += x.size(0)
        loop.set_postfix(loss=f'{run_loss/max(run_total,1):.3f}',
                         acc=f'{100*run_correct/max(run_total,1):.1f}%')

    scheduler.step()
    tr_loss = run_loss / max(run_total, 1)
    tr_acc  = 100.0 * run_correct / max(run_total, 1)

    # ---- VALIDATE ----
    model.eval()
    vl_loss = vl_correct = vl_total = 0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            if use_amp:
                with torch.amp.autocast('cuda'):
                    out  = model(x)
                    loss = criterion(out, y)
            else:
                out  = model(x)
                loss = criterion(out, y)
            if not torch.isnan(loss):
                vl_loss += loss.item() * x.size(0)
            vl_correct += out.argmax(1).eq(y).sum().item()
            vl_total   += x.size(0)

    vl_loss = vl_loss / max(vl_total, 1)
    vl_acc  = 100.0 * vl_correct / max(vl_total, 1)

    history['train_acc'].append(tr_acc)
    history['val_acc'].append(vl_acc)
    history['train_loss'].append(tr_loss)
    history['val_loss'].append(vl_loss)

    flag = ''
    if vl_acc > best_val_acc:
        best_val_acc = vl_acc
        torch.save({
            'model': model.state_dict(),
            'epoch': epoch,
            'val_acc': vl_acc,
            'num_classes': num_classes,
            'class_labels': class_labels,
            'class_map': class_map,
        }, BEST_PATH)
        flag = '  🏆 best'

    # Rolling checkpoint (resume in case of timeout)
    torch.save({
        'model': model.state_dict(),
        'optimizer': optimizer.state_dict(),
        'scheduler': scheduler.state_dict(),
        'epoch': epoch,
        'best_val_acc': best_val_acc,
        'num_classes': num_classes,
    }, LATEST_PATH)

    lr = optimizer.param_groups[0]['lr']
    print(f'E{epoch+1:02d}/{EPOCHS} | '
          f'Train {tr_acc:.2f}% loss={tr_loss:.4f} | '
          f'Val {vl_acc:.2f}% loss={vl_loss:.4f} | '
          f'LR={lr:.1e} | {time.time()-t0:.0f}s{flag}')

print(f'\n🎉 Done! Best val acc: {best_val_acc:.2f}%')
print(f'📂 Download: {BEST_PATH}')


🔥 Training 15 epochs  |  batch=8  |  AMP=True


E01/15 | Train 48.91% loss=2.4869 | Val 61.30% loss=2.0274 | LR=3.0e-04 | 96s  🏆 best


E02/15 | Train 72.17% loss=1.6369 | Val 67.02% loss=2.0056 | LR=2.9e-04 | 104s  🏆 best


E03/15 | Train 80.61% loss=1.4062 | Val 73.71% loss=1.7319 | LR=2.7e-04 | 104s  🏆 best


E04/15 | Train 85.59% loss=1.2655 | Val 77.50% loss=1.5614 | LR=2.5e-04 | 104s  🏆 best


...

E11/15 | Train 99.31% loss=0.8268 | Val 87.10% loss=1.2058 | LR=2.7e-05 | 141s


E12/15 | Train 99.70% loss=0.8098 | Val 88.94% loss=1.1773 | LR=2.0e-05 | 142s  🏆 best


E13/15 | Train 99.76% loss=0.8047 | Val 88.46% loss=1.1680 | LR=1.0e-05 | 141s


E14/15 | Train 99.81% loss=0.8020 | Val 88.26% loss=1.1638 | LR=3.0e-06 | 141s


E15/15 | Train 99.89% loss=0.7958 | Val 88.65% loss=1.1502 | LR=1.0e-07 | 141s

🎉 Done! Best val acc: 88.94%
📂 Download: /kaggle/working/best_r2plus1d_qevd.pth


In [9]:
# =========================================================
# TRAINING CURVES
# =========================================================
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
ep = range(1, len(history['train_acc']) + 1)
ax1.plot(ep, history['train_acc'], 'b-o', ms=4, label='Train')
ax1.plot(ep, history['val_acc'],   'r-o', ms=4, label='Val')
ax1.axhline(90, color='g', ls='--', alpha=0.6, label='90% target')
ax1.set(xlabel='Epoch', ylabel='Accuracy (%)', title='Accuracy')
ax1.legend(); ax1.grid(alpha=0.3)
ax2.plot(ep, history['train_loss'], 'b-o', ms=4, label='Train')
ax2.plot(ep, history['val_loss'],   'r-o', ms=4, label='Val')
ax2.set(xlabel='Epoch', ylabel='Loss', title='Loss')
ax2.legend(); ax2.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'training_curves.png'), dpi=150)
plt.show()
print('📈 curves saved')

📈 curves saved


In [10]:
# =========================================================
# VERIFY BEST MODEL (clean reload)
# =========================================================
verify = r2plus1d_18(weights=None)
verify.fc = nn.Linear(verify.fc.in_features, num_classes)
ckpt = torch.load(BEST_PATH, map_location='cpu', weights_only=False)
verify.load_state_dict(ckpt['model'])
verify = verify.to(device).eval()

correct = total = 0
with torch.no_grad():
    for x, y in tqdm(val_loader, desc='Final val'):
        out = verify(x.to(device))
        correct += out.argmax(1).eq(y.to(device)).sum().item()
        total   += y.size(0)

print(f'\n✅ Final Val Acc: {100*correct/max(total,1):.2f}%')

# Output shape sanity check
dummy = torch.randn(1, 3, 16, 112, 112).to(device)
with torch.no_grad():
    out = verify(dummy)
print(f'✅ Output shape: {out.shape}  (expect (1, {num_classes}))')
print('\n🚀 Download best_r2plus1d_qevd.pth → run Export notebook locally')

Final val: 100%|██████████| 129/129 [00:43<00:00,  2.96it/s]


✅ Final Val Acc: 88.94%
✅ Output shape: torch.Size([1, 91])  (expect (1, 91))

🚀 Download best_r2plus1d_qevd.pth → run Export notebook locally


In [13]:
!pip install onnx onnxscript onnxruntime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 689.1/689.1 kB 10.8 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 73.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.1/164.1 kB 13.0 MB/s eta 0:00:00


In [16]:
import os
import torch
import torch.nn as nn
from torchvision.models.video import r2plus1d_18

# =========================================================
# 1. Configuration (MUST match your 03_training notebook)
# =========================================================
MODEL_PATH  = "/kaggle/working/best_r2plus1d_qevd.pth"
ONNX_PATH   = "/kaggle/working/qualcomm_r2plus1d.onnx"
NUM_CLASSES = 91  # Verified from your previous size mismatch error
BATCH_SIZE  = 1
NUM_FRAMES  = 16
HEIGHT, WIDTH = 112, 112

# =========================================================
# 2. Recreate and Load Model
# =========================================================
print(f"🔄 Loading weights from {MODEL_PATH}...")
model = r2plus1d_18(weights=None)
model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)

# Load the checkpoint
if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(f"❌ Checkpoint not found at {MODEL_PATH}. Did training finish?")

ckpt = torch.load(MODEL_PATH, map_location="cpu")

# Handle different saving formats (wrapper dict vs state_dict)
if isinstance(ckpt, dict) and "model" in ckpt:
    model.load_state_dict(ckpt["model"])
elif isinstance(ckpt, dict) and "state_dict" in ckpt:
    model.load_state_dict(ckpt["state_dict"])
else:
    model.load_state_dict(ckpt)

model.to('cpu')
model.eval()

# =========================================================
# 3. Create Dummy Input
# =========================================================
# Shape: (Batch, Channels, Time/Frames, Height, Width)
dummy_input = torch.randn(BATCH_SIZE, 3, NUM_FRAMES, HEIGHT, WIDTH).to('cpu')

# =========================================================
# 4. Export to ONNX (Optimized for Qualcomm AI Hub)
# =========================================================
print(f"🚀 Exporting to {ONNX_PATH}...")

try:
    torch.onnx.export(
        model,
        dummy_input,
        ONNX_PATH,
        export_params=True,        # ESSENTIAL: Embeds weights in the file
        opset_version=15,          # Higher opset to avoid the "axes_input" crash
        do_constant_folding=True,  # Folds batch norm and constants for NPU
        input_names=['input'],     
        output_names=['output'],
        # Crucial for Qualcomm: ensures weights aren't saved as separate files
        keep_initializers_as_inputs=False, 
        dynamic_axes=None          # Fixed shapes = Faster NPU performance
    )
    
    # Final Verification of File Size
    size_mb = os.path.getsize(ONNX_PATH) / (1024 * 1024)
    if size_mb > 50:
        print(f"✅ ONNX export complete! Size: {size_mb:.2f} MB")
        print(f"📍 File location: {ONNX_PATH}")
    else:
        print(f"⚠️ Warning: File size is {size_mb:.2f} MB. This is likely too small!")
        
except Exception as e:
    print(f"❌ Export failed: {e}")

🔄 Loading weights from /kaggle/working/best_r2plus1d_qevd.pth...


W0409 15:09:24.897000 55 torch/onnx/_internal/exporter/_compat.py:125] Setting ONNX exporter to use operator set version 18 because the requested opset_version 15 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


🚀 Exporting to /kaggle/working/qualcomm_r2plus1d.onnx...


W0409 15:09:25.388000 55 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0409 15:09:25.390000 55 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0409 15:09:25.391000 55 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.
W0409 15:09:25.393000 55 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.


[torch.onnx] Obtain model graph for `VideoResNet([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `VideoResNet([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...


The model version conversion is not supported by the onnxscript version converter and fallback is enabled. The model will be converted using the onnx C API (target version: 15).
Failed to convert the model to the target version 15 using the ONNX C API. The model was not modified
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/__init__.py", line 120, in call
    converted_proto = _c_api_utils.call_onnx_api(
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
             ^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/__init__.py", line 115, in _partial_convert_version
    return onnx.version_converter.convert_version(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnx/version_converter.py"

[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
Applied 74 of general pattern rewrite rules.
⚠️ Warning: File size is 0.17 MB. This is likely too small!


In [20]:
import os
import torch
import torch.nn as nn
from torchvision.models.video import r2plus1d_18

# 1. Setup paths
MODEL_PATH  = "/kaggle/working/best_r2plus1d_qevd.pth"
ONNX_PATH   = "/kaggle/working/qualcomm_r2plus1d.onnx"
NUM_CLASSES = 91 

# 2. Re-initialize and load weights correctly
print(f"🔄 Loading weights from {MODEL_PATH}...")
model = r2plus1d_18(weights=None)
model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)

ckpt = torch.load(MODEL_PATH, map_location="cpu")
state_dict = ckpt["model"] if "model" in ckpt else ckpt
model.load_state_dict(state_dict)
model.eval()

# 3. Create dummy input
dummy_input = torch.randn(1, 3, 16, 112, 112)

# 4. THE FIX: Explicitly disable the Dynamo engine
print(f"🚀 Exporting using Legacy Path (Bypassing Dynamo)...")

# By setting dynamo=False and a specific operator_export_type, 
# we force the old, reliable exporter to handle the weights.
torch.onnx.export(
    model,
    dummy_input,
    ONNX_PATH,
    export_params=True,
    opset_version=14,           # Opset 14 is very stable for Qualcomm
    do_constant_folding=True,
    input_names=['input'],
    output_names=['output'],
    # CRITICAL: This argument forces the legacy path in PyTorch 2.x
    dynamo=False,               
    keep_initializers_as_inputs=False
)

# 5. Final verification
if os.path.exists(ONNX_PATH):
    size_mb = os.path.getsize(ONNX_PATH) / (1024 * 1024)
    print(f"📏 Final ONNX Size: {size_mb:.2f} MB")
    if size_mb > 100:
        print("✅ SUCCESS! Weights are embedded.")
    else:
        print("❌ Size still too small. Attempting emergency serialization...")
        # If this still fails, the model object itself might be corrupted in memory.
        # Try restarting the Kaggle session and running only this block.

🔄 Loading weights from /kaggle/working/best_r2plus1d_qevd.pth...
🚀 Exporting using Legacy Path (Bypassing Dynamo)...
📏 Final ONNX Size: 119.55 MB
✅ SUCCESS! Weights are embedded.


In [21]:
import os
file_path = "/kaggle/working/qualcomm_r2plus1d.onnx"
if os.path.exists(file_path):
    size_mb = os.path.getsize(file_path) / (1024 * 1024)
    print(f"✅ Found ONNX file!")
    print(f"📏 Size: {size_mb:.2f} MB")
else:
    print("❌ File not found. Check the output path.")

✅ Found ONNX file!
📏 Size: 119.55 MB


In [23]:
import onnx
import os

# 1. Define your current paths
input_onnx = "/kaggle/working/qualcomm_r2plus1d.onnx"
input_data = "/kaggle/working/qualcomm_r2plus1d.onnx.data"
output_unified = "/kaggle/working/lpcvc_final_unified.onnx"

if not os.path.exists(input_onnx):
    print(f"❌ Could not find {input_onnx}")
else:
    # 2. Load the model
    # This automatically finds the .data file if it's in the same folder
    print("🔄 Loading split model and weights...")
    model = onnx.load(input_onnx)

    # 3. Save as a single file
    # save_as_external_data=False forces the weights INTO the .onnx file
    print("🔗 Merging into a single file...")
    onnx.save(
        model, 
        output_unified, 
        save_as_external_data=False
    )

    # 4. Verify the results
    new_size = os.path.getsize(output_unified) / (1024 * 1024)
    print(f"✅ SUCCESS! Unified Model created: {output_unified}")
    print(f"📏 Final Size: {new_size:.2f} MB")

    # 5. Clean up (Optional)
    # You can now delete the split files to save Kaggle space
    # os.remove(input_onnx)
    # os.remove(input_data)

🔄 Loading split model and weights...
🔗 Merging into a single file...
✅ SUCCESS! Unified Model created: /kaggle/working/lpcvc_final_unified.onnx
📏 Final Size: 119.55 MB
